# Preprocessing: Build Matchup Training Data

This notebook builds the training dataset for the pitcher-batter matchup model.

**Each row = one plate appearance** with:
1. Pitcher historical profile (season stats + per-pitch-type arsenal)
2. Pitcher rolling stats (by last 5/10/20 starts)
3. Batter historical profile (season stats + vs-pitch-type performance)
4. Batter rolling stats (by last 25/50/100 games)
5. Handedness encoding
6. Target outcome (K, BB, 1B, 2B, 3B, HR, OUT)

**Rolling stats include:** whiff%, csw%, K%, BB%, zone%, chase%, xwOBA, exit velo, barrel%, hard hit%

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_batter_pitch_stats,
)

from src.data.preprocess import (
    MatchupPreprocessor,
    prepare_temporal_split,
    OUTCOME_CLASSES,
)

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

# Date range
START_DATE = '2024-05-01'
END_DATE = '2024-05-05'

# For splitting
TEST_DATE = '2024-06-15'
VAL_DATE = '2024-06-01'

# Rolling windows
PITCHER_ROLLING_WINDOWS = [5, 10, 20]  # By starts
BATTER_ROLLING_WINDOWS = [25, 50, 100]  # By games

## 1. Load Raw Data

Load the Statcast pitches and pre-computed profiles from notebook 02.

In [2]:
# Option A: Load from saved CSVs (faster if already run notebook 02)
from pathlib import Path

data_dir = Path('../data/statcast')

if (data_dir / 'pitcher_arsenal.csv').exists():
    print("Loading from saved CSVs...")
    pitcher_profiles = pd.read_csv(data_dir / 'pitcher_arsenal.csv')
    batter_profiles = pd.read_csv(data_dir / 'batter_profiles.csv')
    print(f"Pitcher profiles: {len(pitcher_profiles)}")
    print(f"Batter profiles: {len(batter_profiles)}")
else:
    print("No saved data found. Will compute from scratch.")

Loading from saved CSVs...
Pitcher profiles: 380
Batter profiles: 388


In [3]:
# Fetch raw Statcast pitches (needed for at-bat features and rolling stats)
pitches = get_statcast_pitches(START_DATE, END_DATE)
print(f"Total pitches: {len(pitches):,}")

Fetching Statcast pitches from 2024-05-01 to 2024-05-05...
  2024-05-01 to 2024-05-05...
This is a large query, it may take a moment to complete


100%|██████████| 5/5 [00:12<00:00,  2.42s/it]


  Total pitches: 19,165
Total pitches: 19,165


In [4]:
# Compute profiles if not loaded
if 'pitcher_profiles' not in dir():
    print("Computing pitcher profiles...")
    pitcher_profiles = get_pitcher_arsenal(
        START_DATE, END_DATE,
        min_pitches=200,
        pitches_df=pitches
    )

if 'batter_profiles' not in dir():
    print("Computing batter profiles...")
    batter_profiles = get_batter_pitch_stats(
        START_DATE, END_DATE,
        min_pitches=200,
        pitches_df=pitches
    )

## 2. Build Matchup Data

Combine at-bat features, profiles, and rolling stats into one training dataset.

In [5]:
# Initialize preprocessor
preprocessor = MatchupPreprocessor()

# Build the matchup dataset
matchups = preprocessor.build_matchup_data(
    pitches_df=pitches,
    pitcher_profiles=pitcher_profiles,
    batter_profiles=batter_profiles,
    pitcher_rolling_windows=PITCHER_ROLLING_WINDOWS,
    batter_rolling_windows=BATTER_ROLLING_WINDOWS,
)

print(f"\nMatchup dataset shape: {matchups.shape}")
matchups.head(10)

Building matchup training data...
  Extracting plate appearances...
  Computing pitcher rolling stats...
  Computing batter rolling stats...
  Merging features...
  Adding handedness features...
  Built 4,829 matchup rows

Matchup dataset shape: (4829, 196)


,game_pk,game_date,at_bat_number,pitcher_id,batter_id,p_throws,stand,events,outcome,p_pitcher_id,p_pitcher_name,p_p_throws,p_total_pitches,p_primary_pitch,p_ff_usage_pct,p_ff_velo_avg,p_ff_spin_avg,p_ff_vmov_avg,p_ff_hmov_avg,p_ff_whiff_pct,p_si_usage_pct,p_fc_usage_pct,p_sl_usage_pct,p_cu_usage_pct,p_cu_velo_avg,p_cu_spin_avg,p_cu_vmov_avg,p_cu_hmov_avg,p_cu_whiff_pct,p_ch_usage_pct,p_sv_usage_pct,p_fs_usage_pct,p_kc_usage_pct,p_st_usage_pct,p_kn_usage_pct,p_whiff_pct,p_swstr_pct,p_csw_pct,p_zone_pct,p_chase_pct_induced,...,p_roll20_hard_hit_rate,b_roll25_whiff_rate,b_roll25_contact_rate,b_roll25_k_rate,b_roll25_bb_rate,b_roll25_zone_swing_rate,b_roll25_chase_rate,b_roll25_xwoba,b_roll25_avg_exit_velo,b_roll25_barrel_rate,b_roll25_hard_hit_rate,b_roll50_whiff_rate,b_roll50_contact_rate,b_roll50_k_rate,b_roll50_bb_rate,b_roll50_zone_swing_rate,b_roll50_chase_rate,b_roll50_xwoba,b_roll50_avg_exit_velo,b_roll50_barrel_rate,b_roll50_hard_hit_rate,b_roll100_whiff_rate,b_roll100_contact_rate,b_roll100_k_rate,b_roll100_bb_rate,b_roll100_zone_swing_rate,b_roll100_chase_rate,b_roll100_xwoba,b_roll100_avg_exit_velo,b_roll100_barrel_rate,b_roll100_hard_hit_rate,p_throws_L,p_throws_R,stand_L,stand_R,matchup_LvL,matchup_LvR,matchup_RvL,matchup_RvR,same_hand
0,744861,2024-05-05,1,669022,543807,L,R,strikeout,K,669022.0,"Gore, MacKenzie",L,76.0,FF,0.578947,96.522727,2315.659091,1.480909,0.510227,0.166667,0.000000,0.078947,0.000000,0.236842,NaN,NaN,NaN,NaN,NaN,0.105263,0.0,0.0,0.0,0.0,0.0,0.135135,0.065789,0.236842,0.526316,0.250000,...,0.087222,0.205808,0.794192,0.187778,0.033333,0.784565,0.286714,0.238685,82.819500,0.000000,0.250397,0.214729,0.785271,0.227273,0.043939,0.718787,0.316210,0.272355,83.037803,0.003788,0.243194,0.239680,0.760320,0.230513,0.058462,0.694253,0.342546,0.281744,82.615763,0.009418,0.228596,1,0,0,1,0,1,0,0,0
1,744861,2024-05-05,2,669022,665489,L,R,single,1B,669022.0,"Gore, MacKenzie",L,76.0,FF,0.578947,96.522727,2315.659091,1.480909,0.510227,0.166667,0.000000,0.078947,0.000000,0.236842,NaN,NaN,NaN,NaN,NaN,0.105263,0.0,0.0,0.0,0.0,0.0,0.135135,0.065789,0.236842,0.526316,0.250000,...,0.087222,0.223881,0.776119,0.240196,0.158824,0.627188,0.244199,0.340133,83.454317,0.000000,0.263072,0.212042,0.787958,0.236667,0.110000,0.630212,0.220114,0.333999,84.814586,0.009524,0.331383,0.188527,0.811473,0.228571,0.095000,0.632269,0.224680,0.309783,83.029929,0.013946,0.289722,1,0,0,1,0,1,0,0,0
2,744861,2024-05-05,3,669022,457759,L,R,strikeout,K,669022.0,"Gore, MacKenzie",L,76.0,FF,0.578947,96.522727,2315.659091,1.480909,0.510227,0.166667,0.000000,0.078947,0.000000,0.236842,NaN,NaN,NaN,NaN,NaN,0.105263,0.0,0.0,0.0,0.0,0.0,0.135135,0.065789,0.236842,0.526316,0.250000,...,0.087222,0.243821,0.756179,0.166667,0.062500,0.710985,0.401051,0.346318,84.296621,0.039683,0.287302,0.243821,0.756179,0.166667,0.062500,0.710985,0.401051,0.346318,84.296621,0.039683,0.287302,0.243821,0.756179,0.166667,0.062500,0.710985,0.401051,0.346318,84.296621,0.039683,0.287302,1,0,0,1,0,1,0,0,0
3,744861,2024-05-05,4,669022,666182,L,R,force_out,OUT,669022.0,"Gore, MacKenzie",L,76.0,FF,0.578947,96.522727,2315.659091,1.480909,0.510227,0.166667,0.000000,0.078947,0.000000,0.236842,NaN,NaN,NaN,NaN,NaN,0.105263,0.0,0.0,0.0,0.0,0.0,0.135135,0.065789,0.236842,0.526316,0.250000,...,0.087222,0.212826,0.787174,0.262745,0.039216,0.757041,0.180035,0.302739,87.016176,0.031250,0.304985,0.217234,0.782766,0.224510,0.066176,0.689127,0.256436,0.302378,83.624281,0.015625,0.248227,0.218757,0.781243,0.219324,0.093478,0.641975,0.261053,0.324770,84.981367,0.009595,0.291773,1,0,0,1,0,1,0,0,0
4,744861,2024-05-05,5,666201,696285,R,R,field_error,OUT,666201.0,"Manoah, Alek",R,92.0,SI,0.217391,94.140000,2299.550000,1.347000,-0.469500,0.200000,0.402174,0.000000,0.369565,0.000000,NaN,NaN,NaN,NaN,NaN,0.010870,0.0,0.0,0.0,0.0,0.0,0.205882,0.076087,0.271739,0.467391,0.183673,...,0.173333,0.209303,0.790697,0.249074,0.107407,0.564137,0.231708,0.374673,84.090516,0.011111,0.322884,0.223154,0.776846,0.262381,

In [6]:
# All columns in the matchup dataset
print(f"Total columns: {len(matchups.columns)}")
print("\nColumn groups:")

# Pitcher profile features (season-level)
p_cols = [c for c in matchups.columns if c.startswith('p_') and 'roll' not in c]
print(f"\nPitcher profile features ({len(p_cols)}):  [season-level arsenal stats]")
print(p_cols[:15], '...' if len(p_cols) > 15 else '')

# Pitcher rolling features
p_roll_cols = [c for c in matchups.columns if c.startswith('p_roll')]
print(f"\nPitcher rolling features ({len(p_roll_cols)}):  [by last N starts]")
print(p_roll_cols)

# Batter profile features (season-level)
b_cols = [c for c in matchups.columns if c.startswith('b_') and 'roll' not in c]
print(f"\nBatter profile features ({len(b_cols)}):  [season-level vs pitch types]")
print(b_cols[:15], '...' if len(b_cols) > 15 else '')

# Batter rolling features
b_roll_cols = [c for c in matchups.columns if c.startswith('b_roll')]
print(f"\nBatter rolling features ({len(b_roll_cols)}):  [by last N games]")
print(b_roll_cols)

# Handedness features
hand_cols = [c for c in matchups.columns if 'throw' in c or 'stand' in c or 'matchup' in c or 'hand' in c]
print(f"\nHandedness features ({len(hand_cols)}):")
print(hand_cols)

Total columns: 196

Column groups:

Pitcher profile features (76):  [season-level arsenal stats]
['p_throws', 'p_pitcher_id', 'p_pitcher_name', 'p_p_throws', 'p_total_pitches', 'p_primary_pitch', 'p_ff_usage_pct', 'p_ff_velo_avg', 'p_ff_spin_avg', 'p_ff_vmov_avg', 'p_ff_hmov_avg', 'p_ff_whiff_pct', 'p_si_usage_pct', 'p_fc_usage_pct', 'p_sl_usage_pct'] ...

Pitcher rolling features (33):  [by last N starts]
['p_roll5_whiff_rate', 'p_roll5_csw_rate', 'p_roll5_k_rate', 'p_roll5_bb_rate', 'p_roll5_zone_rate', 'p_roll5_chase_rate', 'p_roll5_avg_velo', 'p_roll5_xwoba', 'p_roll5_avg_exit_velo', 'p_roll5_barrel_rate', 'p_roll5_hard_hit_rate', 'p_roll10_whiff_rate', 'p_roll10_csw_rate', 'p_roll10_k_rate', 'p_roll10_bb_rate', 'p_roll10_zone_rate', 'p_roll10_chase_rate', 'p_roll10_avg_velo', 'p_roll10_xwoba', 'p_roll10_avg_exit_velo', 'p_roll10_barrel_rate', 'p_roll10_hard_hit_rate', 'p_roll20_whiff_rate', 'p_roll20_csw_rate', 'p_roll20_k_rate', 'p_roll20_bb_rate', 'p_roll20_zone_rate', 'p_roll20

In [7]:
# Outcome distribution
print("Outcome distribution:")
outcome_counts = matchups['outcome'].value_counts()
outcome_pcts = matchups['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

Outcome distribution:


,count,pct
outcome,,
OUT,2301,47.6
K,1115,23.1
1B,663,13.7
BB,406,8.4
2B,199,4.1
HR,120,2.5
3B,25,0.5


## 3. Explore Feature Groups

In [8]:
# Pitcher rolling features (by starts)
print("Pitcher rolling features (by last N starts):")
print("Stats: whiff_rate, csw_rate, k_rate, bb_rate, zone_rate, chase_rate,")
print("       avg_velo, xwoba, avg_exit_velo, barrel_rate, hard_hit_rate\n")
matchups[p_roll_cols].describe().round(3)

Pitcher rolling features (by last N starts):
Stats: whiff_rate, csw_rate, k_rate, bb_rate, zone_rate, chase_rate,
       avg_velo, xwoba, avg_exit_velo, barrel_rate, hard_hit_rate



,p_roll5_whiff_rate,p_roll5_csw_rate,p_roll5_k_rate,p_roll5_bb_rate,p_roll5_zone_rate,p_roll5_chase_rate,p_roll5_avg_velo,p_roll5_xwoba,p_roll5_avg_exit_velo,p_roll5_barrel_rate,p_roll5_hard_hit_rate,p_roll10_whiff_rate,p_roll10_csw_rate,p_roll10_k_rate,p_roll10_bb_rate,p_roll10_zone_rate,p_roll10_chase_rate,p_roll10_avg_velo,p_roll10_xwoba,p_roll10_avg_exit_velo,p_roll10_barrel_rate,p_roll10_hard_hit_rate,p_roll20_whiff_rate,p_roll20_csw_rate,p_roll20_k_rate,p_roll20_bb_rate,p_roll20_zone_rate,p_roll20_chase_rate,p_roll20_avg_velo,p_roll20_xwoba,p_roll20_avg_exit_velo,p_roll20_barrel_rate,p_roll20_hard_hit_rate
count,4243.000,4243.000,4243.000,4243.000,4243.000,4243.000,4243.000,4243.000,4234.000,4234.00,4234.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4785.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000,4795.000
mean,0.232,0.283,0.242,0.084,0.497,0.271,89.753,0.264,82.291,0.01,0.233,0.237,0.280,0.246,0.088,0.499,0.276,89.739,0.263,82.145,0.011,0.230,0.239,0.280,0.250,0.084,0.496,0.285,89.602,0.260,81.596,0.012,0.211
std,0.139,0.095,0.196,0.121,0.100,0.161,3.184,0.131,7.105,0.03,0.215,0.101,0.073,0.152,0.096,0.084,0.123,2.463,0.089,5.426,0.026,0.161,0.064,0.049,0.110,0.067,0.062,0.080,1.787,0.057,3.371,0.020,0.093
min,0.000,0.000,0.000,0.000,0.222,0.000,78.400,0.000,60.600,0.00,0.000,0.000,0.000,0.000,0.000,0.278,0.000,80.545,0.013,67.152,0.000,0.000,0.033,0.071,0.000,0.000,0.286,0.096,82.621,0.105,72.843,0.000,0.000
25%,0.138,0.231,0.100,0.000,0.451,0.154,87.859,0.187,77.735,0.00,0.083,0.162,0.249,0.133,0.000,0.454,0.200,88.558,0.201,78.714,0.000,0.125,0.202,0.260,0.167,0.033,0.464,0.225,88.760,0.220,79.471,0.000,0.156
50%,0.227,0.278,0.217,0.000,0.500,0.250,89.629,0.255,81.400,0.00,0.200,0.235,0.280,0.222,0.067,0.500,0.269,89.852,0.261,81.664,0.000,0.206,0.240,0.289,0.249,0.082,0.500,0.278,89.669,0.262,81.994,0.000,0.207
75%,0.325,0.340,0.333,0.167,0.546,0.384,91.799,0.342,86.237,0.00,0.333,0.304,0.320,0.350,0.125,0.535,0.352,91.161,0.318,84.917,0.000,0.321,0.287,0.309,0.327,0.117,0.532,0.339,90.920,0.298,83.580,0.020,0.278
max,0.750,0.545,1.000,0.667,0.875,1.000,97.608,0.925,106.000,0.20,1.000,0.571,0.477,0.833,0.500,0.875,0.688,97.608,0.925,106.000,0.155,1.000,0.388,0.381,0.606,0.417,0.650,0.542,93.012,0.432,89.600,0.103,0.464


In [9]:
# Sample of pitcher arsenal features
pitcher_sample_cols = [
    'p_ff_velo_avg', 'p_ff_spin_avg', 'p_ff_whiff_pct',
    'p_sl_velo_avg', 'p_sl_whiff_pct',
    'p_whiff_pct', 'p_csw_pct', 'p_zone_pct'
]
available = [c for c in pitcher_sample_cols if c in matchups.columns]
print("Pitcher arsenal features sample:")
matchups[available].describe().round(3)

Pitcher arsenal features sample:


,p_ff_velo_avg,p_ff_spin_avg,p_ff_whiff_pct,p_sl_velo_avg,p_sl_whiff_pct,p_whiff_pct,p_csw_pct,p_zone_pct
count,2397.000,2397.000,2397.000,862.000,862.000,4826.000,4826.000,4826.000
mean,93.757,2276.459,0.180,85.929,0.370,0.228,0.278,0.499
std,2.015,150.410,0.138,2.831,0.162,0.096,0.064,0.070
min,85.023,1922.500,0.000,77.807,0.062,0.000,0.000,0.278
25%,92.295,2185.848,0.094,84.965,0.235,0.156,0.243,0.451
50%,93.806,2274.905,0.160,86.425,0.385,0.217,0.278,0.500
75%,95.103,2362.792,0.250,87.800,0.500,0.289,0.310,0.538
max,98.017,2643.226,1.000,90.208,0.667,0.625,0.571,0.800


In [10]:
# Sample of batter profile features
batter_sample_cols = [
    'b_whiff_pct', 'b_contact_pct', 'b_chase_pct',
    'b_vs_ff_whiff_pct', 'b_vs_sl_whiff_pct',
    'b_velo_97_plus_whiff_pct', 'b_avg_exit_velo', 'b_xwoba'
]
available = [c for c in batter_sample_cols if c in matchups.columns]
print("Batter profile features sample:")
matchups[available].describe().round(3)

Batter profile features sample:


,b_whiff_pct,b_contact_pct,b_chase_pct,b_vs_ff_whiff_pct,b_vs_sl_whiff_pct,b_avg_exit_velo,b_xwoba
count,4823.000,4823.000,4823.000,1917.000,291.000,44.000,44.000
mean,0.228,0.772,0.279,0.186,0.343,86.385,0.353
std,0.110,0.110,0.111,0.152,0.184,0.553,0.040
min,0.000,0.250,0.000,0.000,0.000,85.862,0.315
25%,0.156,0.714,0.205,0.071,0.200,85.862,0.315
50%,0.222,0.778,0.267,0.167,0.333,85.862,0.315
75%,0.286,0.844,0.350,0.259,0.500,86.957,0.394
max,0.750,1.000,1.000,0.667,0.714,86.957,0.394


In [11]:
# Batter rolling features (by games)
print("Batter rolling features (by last N games):")
print("Stats: whiff_rate, contact_rate, k_rate, bb_rate, zone_swing_rate, chase_rate,")
print("       xwoba, avg_exit_velo, barrel_rate, hard_hit_rate\n")
matchups[b_roll_cols].describe().round(3)

Batter rolling features (by last N games):
Stats: whiff_rate, contact_rate, k_rate, bb_rate, zone_swing_rate, chase_rate,
       xwoba, avg_exit_velo, barrel_rate, hard_hit_rate



,b_roll25_whiff_rate,b_roll25_contact_rate,b_roll25_k_rate,b_roll25_bb_rate,b_roll25_zone_swing_rate,b_roll25_chase_rate,b_roll25_xwoba,b_roll25_avg_exit_velo,b_roll25_barrel_rate,b_roll25_hard_hit_rate,b_roll50_whiff_rate,b_roll50_contact_rate,b_roll50_k_rate,b_roll50_bb_rate,b_roll50_zone_swing_rate,b_roll50_chase_rate,b_roll50_xwoba,b_roll50_avg_exit_velo,b_roll50_barrel_rate,b_roll50_hard_hit_rate,b_roll100_whiff_rate,b_roll100_contact_rate,b_roll100_k_rate,b_roll100_bb_rate,b_roll100_zone_swing_rate,b_roll100_chase_rate,b_roll100_xwoba,b_roll100_avg_exit_velo,b_roll100_barrel_rate,b_roll100_hard_hit_rate
count,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000,4825.000
mean,0.227,0.773,0.233,0.079,0.658,0.285,0.294,83.110,0.013,0.254,0.227,0.773,0.233,0.079,0.659,0.286,0.293,83.122,0.013,0.253,0.228,0.772,0.234,0.080,0.659,0.288,0.294,83.135,0.013,0.254
std,0.053,0.053,0.066,0.038,0.057,0.065,0.047,2.255,0.015,0.059,0.036,0.036,0.048,0.027,0.045,0.050,0.031,1.535,0.011,0.039,0.024,0.024,0.031,0.017,0.034,0.043,0.018,1.098,0.008,0.028
min,0.093,0.597,0.076,0.000,0.506,0.127,0.160,75.043,0.000,0.072,0.152,0.625,0.083,0.000,0.570,0.184,0.210,78.412,0.000,0.151,0.172,0.625,0.083,0.000,0.594,0.216,0.210,79.460,0.000,0.193
25%,0.193,0.741,0.186,0.054,0.617,0.242,0.262,81.774,0.000,0.218,0.202,0.750,0.206,0.064,0.627,0.255,0.269,81.998,0.004,0.227,0.212,0.755,0.218,0.068,0.632,0.262,0.279,82.476,0.007,0.232
50%,0.221,0.779,0.231,0.078,0.654,0.281,0.294,83.140,0.009,0.252,0.225,0.775,0.232,0.078,0.654,0.283,0.291,83.133,0.010,0.250,0.229,0.771,0.236,0.081,0.656,0.282,0.293,83.113,0.012,0.248
75%,0.259,0.807,0.281,0.102,0.698,0.322,0.328,84.388,0.020,0.287,0.250,0.798,0.260,0.096,0.692,0.313,0.316,84.012,0.020,0.278,0.245,0.788,0.253,0.092,0.683,0.309,0.307,83.839,0.019,0.275
max,0.403,0.907,0.437,0.236,0.822,1.000,0.439,90.819,0.074,0.484,0.375,0.848,0.379,0.157,0.801,1.000,0.380,87.681,0.050,0.388,0.375,0.828,0.316,0.118,0.801,1.000,0.365,86.082,0.046,0.335


In [12]:
# Handedness distribution
print("Handedness matchups:")
matchup_cols = ['matchup_LvL', 'matchup_LvR', 'matchup_RvL', 'matchup_RvR']
available = [c for c in matchup_cols if c in matchups.columns]
matchups[available].sum()

Handedness matchups:


matchup_LvL     337
matchup_LvR     898
matchup_RvL    1776
matchup_RvR    1818
dtype: int64

In [13]:
# Check null percentages
null_pcts = (matchups.isnull().sum() / len(matchups) * 100).sort_values(ascending=False)
print("Columns with >50% nulls:")
print(null_pcts[null_pcts > 50])

print(f"\nTotal columns: {len(matchups.columns)}")
print(f"Columns with any nulls: {(null_pcts > 0).sum()}")
print(f"Columns with >50% nulls: {(null_pcts > 50).sum()}")

Columns with >50% nulls:
b_batter_name          100.000000
b_vs_st_contact_pct     99.647960
b_vs_st_whiff_pct       99.647960
b_vs_ch_xwoba           99.627252
p_kn_hmov_avg           99.606544
                          ...    
p_ff_spin_avg           50.362394
p_ff_hmov_avg           50.362394
p_ff_vmov_avg           50.362394
p_ff_whiff_pct          50.362394
p_ff_velo_avg           50.362394
Length: 83, dtype: float64

Total columns: 196
Columns with any nulls: 178
Columns with >50% nulls: 83


## 4. Preprocess for Model

Scale numeric features and encode target.

In [14]:
# Identify feature columns
numeric_cols, binary_cols = preprocessor.get_feature_columns(matchups)

print(f"Numeric features: {len(numeric_cols)}")
print(f"Binary features: {len(binary_cols)}")
print(f"Total features: {len(numeric_cols) + len(binary_cols)}")

print("\nBinary features:")
print(binary_cols)

Numeric features: 171
Binary features: 9
Total features: 180

Binary features:
['p_throws_L', 'p_throws_R', 'stand_L', 'stand_R', 'matchup_LvL', 'matchup_LvR', 'matchup_RvL', 'matchup_RvR', 'same_hand']


In [ ]:
# Temporal split (avoid leakage)
# Use last ~2 weeks as test, previous ~2 weeks as validation

train_df, test_df, val_df = prepare_temporal_split(
    matchups,
    test_date=TEST_DATE,
    val_date=VAL_DATE,
)

print(f"Train: {len(train_df):,} rows ({train_df['game_date'].min()} to {train_df['game_date'].max()})")
print(f"Val:   {len(val_df):,} rows ({val_df['game_date'].min()} to {val_df['game_date'].max()})")
print(f"Test:  {len(test_df):,} rows ({test_df['game_date'].min()} to {test_df['game_date'].max()})")

Train: 4,829 rows (2024-05-01 00:00:00 to 2024-05-05 00:00:00)
Val:   0 rows (NaT to NaT)
Test:  0 rows (NaT to NaT)


In [16]:
# Fit preprocessor on training data only
preprocessor.fit(train_df)

# Transform all sets
X_train, y_train = preprocessor.transform(train_df, include_target=True)
X_val, y_val = preprocessor.transform(val_df, include_target=True)
X_test, y_test = preprocessor.transform(test_df, include_target=True)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"X_test shape: {X_test.shape}")

print(f"\nFeature names: {len(preprocessor.get_feature_names())}")
print(f"Outcome classes: {preprocessor.get_outcome_classes()}")

ValueError: Found array with 0 sample(s) (shape=(0, 171)) while a minimum of 1 is required by StandardScaler.

In [ ]:
# Check scaled features look reasonable
print("Scaled feature statistics (train):")
print(f"Mean: {np.nanmean(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Std:  {np.nanstd(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Min:  {np.nanmin(X_train, axis=0)[:5].round(2)}... (first 5)")
print(f"Max:  {np.nanmax(X_train, axis=0)[:5].round(2)}... (first 5)")

# Check NaN preservation
nan_count = np.isnan(X_train).sum()
print(f"\nNaN values preserved: {nan_count:,} ({nan_count / X_train.size * 100:.1f}%)")

In [ ]:
# Target distribution in each split
print("Target distribution (%):\n")
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(y, return_counts=True)
    pcts = counts / len(y) * 100
    print(f"{name}:")
    for i, pct in enumerate(pcts):
        print(f"  {preprocessor.get_outcome_classes()[i]}: {pct:.1f}%")
    print()

## 5. Save Preprocessed Data

In [ ]:
from pathlib import Path

# Save matchup DataFrame
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

matchups.to_csv(output_dir / 'matchups.csv', index=False)
print(f"Saved matchups to {output_dir / 'matchups.csv'}")

# Save train/val/test DataFrames
train_df.to_csv(output_dir / 'train.csv', index=False)
val_df.to_csv(output_dir / 'val.csv', index=False)
test_df.to_csv(output_dir / 'test.csv', index=False)
print(f"Saved train/val/test splits")

# Save preprocessor
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)
preprocessor.save(models_dir / 'matchup_preprocessor.pkl')

In [ ]:
# Optionally save numpy arrays for direct model training
np.save(output_dir / 'X_train.npy', X_train)
np.save(output_dir / 'X_val.npy', X_val)
np.save(output_dir / 'X_test.npy', X_test)
np.save(output_dir / 'y_train.npy', y_train)
np.save(output_dir / 'y_val.npy', y_val)
np.save(output_dir / 'y_test.npy', y_test)

print("Saved numpy arrays for model training")

## Summary

The matchup dataset is ready for model training with:

**Features:**
- Pitcher arsenal (per pitch type: velo, spin, movement, usage%, whiff%)
- Pitcher rolling (last 5/10/20 starts): whiff%, csw%, K%, BB%, zone%, chase%, xwOBA, EV, barrel%, hard_hit%
- Batter profile (vs pitch types, velocity buckets, movement profiles)
- Batter rolling (last 25/50/100 games): whiff%, contact%, K%, BB%, zone_swing%, chase%, xwOBA, EV, barrel%, hard_hit%
- Handedness encoding (LvL, LvR, RvL, RvR + individual L/R flags)

**Target:** 7-class outcome (K, BB, 1B, 2B, 3B, HR, OUT)

**Preprocessing:**
- Numeric features scaled (StandardScaler)
- NaN values preserved (for tree models)
- Temporal split (no future leakage)